# Узел А — очистка и запрос классификации

Ноутбук запускает контейнер `data-processor`. Контейнер читает CSV (или создаёт учебный Iris CSV), удаляет пустые и неполные строки, приводит текст к нижнему регистру, передаёт `clean_data.csv` по SSH/SCP и запрашивает прогноз через Flask на узле Б.

Перед запуском заполните `node_a/.env` по инструкции в корневом `README.md`.

## 1. Найти папку узла А и проверить настройки

In [ ]:
from pathlib import Path
import subprocess

current = Path.cwd().resolve()
NODE_A_DIR = current if (current / 'compose.yaml').is_file() and (current / 'process_and_request.py').is_file() else current / 'node_a'
assert (NODE_A_DIR / 'compose.yaml').is_file(), f'Не найдена папка node_a рядом с {current}'
assert (NODE_A_DIR / '.env').is_file(), 'Скопируйте node_a/.env.example в node_a/.env и заполните IP, пользователя, SSH_DIR и NODE_B_DATA_DIR.'
print(f'Папка узла А: {NODE_A_DIR}')

## 2. Собрать и запустить контейнер очистки

In [ ]:
subprocess.run(['docker', 'compose', 'up', '--build', '-d'], cwd=NODE_A_DIR, check=True)
subprocess.run(['docker', 'ps', '--filter', 'name=data-processor'], check=True)

## 3. Очистить, передать и получить ответ

Если `node_a/data/raw_data.csv` существует, контейнер очистит его. Если файла нет, он создаст `raw_data.csv` из встроенного датасета Iris. Скрипт отправит очищенный файл на узел Б, затем вызовет `GET /predict`.

In [ ]:
result = subprocess.run(
    ['docker', 'exec', 'data-processor', 'python', '/app/process_and_request.py'],
    check=True, capture_output=True, text=True
)
if result.stderr:
    print(result.stderr)
print(result.stdout)

Успешный ответ содержит `status: success`, размер обучающей и проверочной частей, точность и прогнозы для проверочных строк. `row_index` — номер строки в очищенном CSV, начиная с нуля.